# dcgan-normal-init-002 — worked example 3: Init only ConvTranspose2d layers (generator-side) and leave encoder Conv2d alone

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-normal-init-002`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Sometimes you want to re-init only the generator's `ConvTranspose2d` upsampling layers with `N(0, 0.02)` while leaving downsampling `Conv2d` encoder layers at their existing values. This is a targeted variant of the DCGAN init: the type check narrows to a single class so `model.apply` only fires on transposed convs.

## Worked solution

**Step 1 — narrow the type check.** Unlike the full DCGAN init that matches both conv types, here `init_fn` matches ONLY `nn.ConvTranspose2d`. A plain `nn.Conv2d` is not an instance of `nn.ConvTranspose2d` (they are siblings, not parent/child), so Conv2d layers are untouched.

**Step 2 — capture the 'before' state.** To prove Conv2d was left alone, we clone its weight before applying the init. Cloning detaches a snapshot that won't change when we mutate in place.

**Step 3 — apply.** `model.apply(init_fn)` walks every submodule; only ConvTranspose2d weights get resampled from `N(0, 0.02)`.

**Step 4 — verify both halves.** The ConvTranspose2d weight std should now be ~0.02. The Conv2d weight should be byte-for-byte equal to its pre-init clone (`torch.equal` is True), proving the selective init did not leak into the encoder.

**Step 5 — reseed.** Because we draw randomness, `t.manual_seed(0)` runs first for reproducibility.

In [ ]:
def init_convt_only(model: nn.Module) -> nn.Module:
    t.manual_seed(0)
    def init_fn(m):
        if isinstance(m, nn.ConvTranspose2d):
            nn.init.normal_(m.weight, 0.0, 0.02)
    model.apply(init_fn)
    return model

t.manual_seed(123)
net = nn.Sequential(
    nn.Conv2d(3, 8, 3, 2, 1),
    nn.ConvTranspose2d(8, 3, 4, 2, 1),
)
before_conv = net[0].weight.clone()
init_convt_only(net)
print('convt std        :', round(net[1].weight.std().item(), 4))
print('conv2d untouched :', t.equal(net[0].weight, before_conv))